# Code Generator

The requirement: use a Frontier model to generate high performance C++ code from Python code


<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">Reminder: OPTIONAL to execute C++ code or Rust code</h2>
            <span style="color:#f71;">As an alternative, you can run it on the website given yesterday</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h1 style="color:#900;">Important Note</h1>
            <span style="color:#900;">
            In this lab, I use high end models GPT 5, Claude 4.5 Sonnet, Gemini 2.5 Pro, Grok 4, which are the slightly higher priced models. The costs are still low, but if you'd prefer to keep costs ultra low, please pick lower cost models like gpt-5-nano.
            </span>
        </td>
    </tr>
</table>

lms server start --port 1234

either one or the other following:

lms load google/gemma-4-31b --context-length 32768

lms load Qwen/Qwen3.5-35B-A3B --context-length 32768

lms load openai/gpt-oss-120b --context-length 32768


In [1]:
# imports

import os
import io
import sys
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
import subprocess
from IPython.display import Markdown, display


In [2]:
load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')
grok_api_key = os.getenv('GROK_API_KEY')
groq_api_key = os.getenv('GROQ_API_KEY')
openrouter_api_key = os.getenv('OPENROUTER_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
    
if anthropic_api_key:
    print(f"Anthropic API Key exists and begins {anthropic_api_key[:7]}")
else:
    print("Anthropic API Key not set (and this is optional)")

if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:2]}")
else:
    print("Google API Key not set (and this is optional)")

if grok_api_key:
    print(f"Grok API Key exists and begins {grok_api_key[:4]}")
else:
    print("Grok API Key not set (and this is optional)")

if groq_api_key:
    print(f"Groq API Key exists and begins {groq_api_key[:4]}")
else:
    print("Groq API Key not set (and this is optional)")

if openrouter_api_key:
    print(f"OpenRouter API Key exists and begins {openrouter_api_key[:6]}")
else:
    print("OpenRouter API Key not set (and this is optional)")



OpenAI API Key exists and begins sk-proj-
Anthropic API Key exists and begins sk-ant-
Google API Key exists and begins AI
Grok API Key exists and begins xai-
Groq API Key exists and begins gsk_
OpenRouter API Key exists and begins sk-or-


In [3]:
# Connect to LM Studio (OpenAI-compatible local API)

from openai import APIConnectionError

MODEL_GEMMA4_31B = "google/gemma-4-31b:2"
MODEL_QWEN35_35B_A3B = "qwen/qwen3.5-35b-a3b"
MODEL_GPT_OSS_120B = "openai/gpt-oss-120b"

LM_STUDIO_CANDIDATE_URLS = [
    os.getenv("LM_STUDIO_URL", "").strip(),
    "http://localhost:1234/v1",
    "http://127.0.0.1:1234/v1",
]
LM_STUDIO_CANDIDATE_URLS = [u for i, u in enumerate(LM_STUDIO_CANDIDATE_URLS) if u and u not in LM_STUDIO_CANDIDATE_URLS[:i]]

def connect_lm_studio():
    errors = []
    for url in LM_STUDIO_CANDIDATE_URLS:
        try:
            client = OpenAI(base_url=url, api_key="lm-studio")
            client.models.list()
            return client, url
        except Exception as e:
            errors.append(f"{url} -> {type(e).__name__}: {e}")

    details = "\n".join(errors) if errors else "No LM Studio URLs configured."
    raise RuntimeError(
        "Could not connect to LM Studio. Start LM Studio and enable the Local Server (Developer tab).\n"
        f"Tried URLs:\n{details}"
    )

lm_studio, LM_STUDIO_URL = connect_lm_studio()
print(f"Connected to LM Studio at {LM_STUDIO_URL}")

def loaded_lm_studio_models():
    try:
        return [m.id for m in lm_studio.models.list().data]
    except Exception as e:
        raise RuntimeError(
            "Failed to query loaded models from LM Studio. Make sure Local Server is running."
        ) from e



Connected to LM Studio at http://localhost:1234/v1


In [4]:
models = [MODEL_GEMMA4_31B, MODEL_QWEN35_35B_A3B, MODEL_GPT_OSS_120B]

clients = {model_name: lm_studio for model_name in models}

print("Configured LM Studio models:", ", ".join(models))
print("Currently loaded in LM Studio:", ", ".join(loaded_lm_studio_models()) or "<none>")



Configured LM Studio models: google/gemma-4-31b:2, qwen/qwen3.5-35b-a3b, openai/gpt-oss-120b
Currently loaded in LM Studio: openai/gpt-oss-120b, google/gemma-4-31b, qwen/qwen3.5-35b-a3b, qwen/qwen3-coder-next, text-embedding-nomic-embed-text-v1.5


In [5]:
from system_info import retrieve_system_info, rust_toolchain_info

system_info = retrieve_system_info()
rust_info = rust_toolchain_info()
rust_info

{'installed': True,
 'rustc': {'path': '/home/peterstroessler/.cargo/bin/rustc',
  'version': 'rustc 1.95.0 (59807616e 2026-04-14)',
  'host_triple': 'aarch64-unknown-linux-gnu',
  'release': '1.95.0',
  'commit_hash': '59807616e1fa2540724bfbac14d7976d7e4a3860'},
 'cargo': {'path': '/home/peterstroessler/.cargo/bin/cargo',
  'version': 'cargo 1.95.0 (f2d3ce0bd 2026-03-21)'},
 'rustup': {'path': '/home/peterstroessler/.cargo/bin/rustup',
  'version': 'rustup 1.29.0 (28d1352db 2026-03-05)',
  'active_toolchain': 'stable-aarch64-unknown-linux-gnu (default)',
  'default_toolchain': '',
  'toolchains': ['stable-aarch64-unknown-linux-gnu (active, default)'],
  'targets_installed': ['aarch64-unknown-linux-gnu']},
 'rust_analyzer': {'path': '/home/peterstroessler/.cargo/bin/rust-analyzer'},
 'env': {'CARGO_HOME': '/home/peterstroessler/.cargo',
  'RUSTUP_HOME': '/home/peterstroessler/.rustup',
  'RUSTFLAGS': '',
  'CARGO_BUILD_TARGET': ''},
 'execution_examples': ['"/home/peterstroessler/.carg

In [6]:
message = f"""
Here is a report of the system information for my computer.
I want to run a Rust compiler to compile a single rust file called main.rs and then execute it in the simplest way possible.
Please reply with whether I need to install a Rust toolchain to do this. If so, please provide the simplest step by step instructions to do so.

If I'm already set up to compile Rust code, then I'd like to run something like this in Python to compile and execute the code:
```python
compile_command = # something here - to achieve the fastest possible runtime performance
compile_result = subprocess.run(compile_command, check=True, text=True, capture_output=True)
run_command = # something here
run_result = subprocess.run(run_command, check=True, text=True, capture_output=True)
return run_result.stdout
```
Please tell me exactly what I should use for the compile_command and run_command.
Have the maximum possible runtime performance in mind; compile time can be slow. Fastest possible runtime performance for this platform is key.
Reply with the commands in markdown.

System information:
{system_info}

Rust toolchain information:
{rust_info}
"""

try:
    response = clients[models[0]].chat.completions.create(
        model=models[0],
        messages=[{"role": "user", "content": message}],
    )
    display(Markdown(response.choices[0].message.content))
except Exception as e:
    display(Markdown(
        "### LM Studio connection/model error\n"
        f"`{type(e).__name__}: {e}`\n\n"
        f"Connected endpoint: `{LM_STUDIO_URL}`\n\n"
        "Open LM Studio, start Local Server, then load one of: "
        f"`{MODEL_GEMMA4_31B}`, `{MODEL_QWEN35_35B_A3B}`, `{MODEL_GPT_OSS_120B}`."
    ))



**Do you need to install a Rust tool‑chain?**  
No – the system already has a working Rust installation:

```
rustc 1.95.0 (aarch64-unknown-linux-gnu)
cargo 1.95.0
rustup 1.29.0
```

So you can compile and run `main.rs` right away.

---

## Fastest‑runtime compilation

For a single source file the simplest way to get the *best possible runtime performance* is to invoke **`rustc` directly** with release‑mode optimisations, link‑time optimisation (LTO), a single code‑gen unit and abort‑on‑panic.  
That gives you an executable that runs as fast as a Cargo “release” build would, but without the extra Cargo overhead.

### Compile command

```python
compile_command = [
    "/home/peterstroessler/.cargo/bin/rustc",   # rustc binary
    "main.rs",                                 # source file
    "-C", "opt-level=3",                       # highest optimisation level
    "-C", "lto=fat",                           # link‑time optimisation (fat LTO)
    "-C", "codegen-units=1",                   # single code‑gen unit → better optimisation
    "-C", "panic=abort",                       # smaller & faster panic handling
    "-C", "target-cpu=native",                 # optimise for this CPU
    "-C", "debuginfo=0",                       # no debug info (smaller binary)
    "-C", "strip=symbols",                     # strip symbols from the final exe
    "-o", "main"                               # output executable name
]
```

*Explanation of the flags*

| Flag | Reason |
|------|--------|
| `-C opt-level=3` | Highest LLVM optimisation level. |
| `-C lto=fat` | Whole‑program optimisation at link time. |
| `-C codegen-units=1` | Forces the compiler to treat the whole crate as one unit, enabling better cross‑function optimisation. |
| `-C panic=abort` | Removes unwind tables – faster and smaller binary when you don’t need unwinding. |
| `-C target-cpu=native` | Generates code tuned for the exact CPU of this machine (aarch64). |
| `-C debuginfo=0` / `-C strip=symbols` | Removes debug information to keep the binary minimal and fast to load. |

### Run command

```python
run_command = ["./main"]          # execute the compiled binary in the current directory
```

---

## Full Python snippet

Below is a self‑contained example that compiles **once** (the compile step may be slow) and then runs the resulting program, returning its standard output.

```python
import subprocess
from pathlib import Path

# ----------------------------------------------------------------------
# 1️⃣ Compile `main.rs` with maximum runtime performance
# ----------------------------------------------------------------------
compile_command = [
    "/home/peterstroessler/.cargo/bin/rustc",
    "main.rs",
    "-C", "opt-level=3",
    "-C", "lto=fat",
    "-C", "codegen-units=1",
    "-C", "panic=abort",
    "-C", "target-cpu=native",
    "-C", "debuginfo=0",
    "-C", "strip=symbols",
    "-o", "main"
]

compile_result = subprocess.run(
    compile_command,
    check=True,          # raise CalledProcessError on failure
    text=True,
    capture_output=True  # you can inspect .stdout/.stderr if needed
)

print("✅ Compilation succeeded")
# ----------------------------------------------------------------------
# 2️⃣ Execute the produced binary
# ----------------------------------------------------------------------
run_command = ["./main"]

run_result = subprocess.run(
    run_command,
    check=True,
    text=True,
    capture_output=True
)

print("🖥️ Program output:")
print(run_result.stdout)
```

*If you need to compile *again* (e.g., after changing `main.rs`), just re‑run the same snippet – the compiler will rebuild the binary with the same optimisation settings.*

---

### TL;DR commands (copy‑paste)

```bash
# 1️⃣ Compile (fastest runtime)
rustc main.rs \
    -C opt-level=3 \
    -C lto=fat \
    -C codegen-units=1 \
    -C panic=abort \
    -C target-cpu=native \
    -C debuginfo=0 \
    -C strip=symbols \
    -o main

# 2️⃣ Run
./main
```

You’re all set – no additional installation is required. Enjoy the maximum performance on your aarch64 Ubuntu 24.04 system!

## For C++, overwrite this with the commands from yesterday, or for Rust, use the new commands

Or just use the website like yesterday:

 https://www.programiz.com/cpp-programming/online-compiler/

In [7]:
compile_command = [
    "/home/peterstroessler/.cargo/bin/rustc",
    "main.rs",
    "-o", "main",
    "-C", "opt-level=3",
    "-C", "target-cpu=native",
    "-C", "lto=fat",
    "-C", "codegen-units=1",
    "-C", "panic=abort",
]

run_command = ["./main"]


## And now, on with the main task

In [8]:
language = "Rust" # or "C++"
extension = "rs" if language == "Rust" else "cpp"

system_prompt = f"""
Your task is to convert Python code into high performance {language} code.
Respond only with {language} code. Do not provide any explanation other than occasional comments.
The {language} response needs to produce an identical output in the fastest possible time.
"""

def user_prompt_for(python):
    return f"""
Port this Python code to {language} with the fastest possible implementation that produces identical output in the least time.
The system information is:
{system_info}
Your response will be written to a file called main.{language} and then compiled and executed; the compilation command is:
{compile_command}
Respond only with {language} code.
Python code to port:

```python
{python}
```
"""

In [9]:
def messages_for(python):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_for(python)}
    ]
 

In [10]:
def write_output(code):
    with open(f"main.{extension}", "w") as f:
        f.write(code)

In [11]:
def port(model, python):
    client = clients[model]

    try:
        loaded = set(loaded_lm_studio_models())
    except Exception as e:
        return (
            f"LM Studio connection error: {type(e).__name__}: {e}\n"
            f"Endpoint: {LM_STUDIO_URL}. Start LM Studio Local Server and retry."
        )

    if model not in loaded:
        loaded_display = ", ".join(sorted(loaded)) if loaded else "<none>"
        return (
            f"Model '{model}' is not currently loaded in LM Studio.\n"
            f"Loaded models: {loaded_display}.\n"
            "In LM Studio, unload the current model and load the selected one, then try again."
        )

    try:
        response = client.chat.completions.create(model=model, messages=messages_for(python))
    except Exception as e:
        return (
            f"LM Studio request failed: {type(e).__name__}: {e}\n"
            f"Endpoint: {LM_STUDIO_URL}."
        )

    reply = response.choices[0].message.content
    reply = reply.replace('```cpp','').replace('```rust','').replace('```','')
    return reply



In [12]:
def run_python(code):
    globals_dict = {"__builtins__": __builtins__}

    buffer = io.StringIO()
    old_stdout = sys.stdout
    sys.stdout = buffer

    try:
        exec(code, globals_dict)
        output = buffer.getvalue()
    except Exception as e:
        output = f"Error: {e}"
    finally:
        sys.stdout = old_stdout

    return output

In [13]:
# Use the commands from GPT 5

def compile_and_run(code):
    write_output(code)
    try:
        subprocess.run(compile_command, check=True, text=True, capture_output=True)
        run_result = subprocess.run(run_command, check=True, text=True, capture_output=True)
        return run_result.stdout
    except subprocess.CalledProcessError as e:
        return f"An error occurred:\n{e.stderr}"

In [14]:
python_hard = """# Be careful to support large numbers

def lcg(seed, a=1664525, c=1013904223, m=2**32):
    value = seed
    while True:
        value = (a * value + c) % m
        yield value
        
def max_subarray_sum(n, seed, min_val, max_val):
    lcg_gen = lcg(seed)
    random_numbers = [next(lcg_gen) % (max_val - min_val + 1) + min_val for _ in range(n)]
    max_sum = float('-inf')
    for i in range(n):
        current_sum = 0
        for j in range(i, n):
            current_sum += random_numbers[j]
            if current_sum > max_sum:
                max_sum = current_sum
    return max_sum

def total_max_subarray_sum(n, initial_seed, min_val, max_val):
    total_sum = 0
    lcg_gen = lcg(initial_seed)
    for _ in range(20):
        seed = next(lcg_gen)
        total_sum += max_subarray_sum(n, seed, min_val, max_val)
    return total_sum

# Parameters
n = 10000         # Number of random numbers
initial_seed = 42 # Initial seed for the LCG
min_val = -10     # Minimum value of random numbers
max_val = 10      # Maximum value of random numbers

# Timing the function
import time
start_time = time.time()
result = total_max_subarray_sum(n, initial_seed, min_val, max_val)
end_time = time.time()

print("Total Maximum Subarray Sum (20 runs):", result)
print("Execution Time: {:.6f} seconds".format(end_time - start_time))
"""

In [15]:
from styles import CSS

with gr.Blocks(css=CSS, theme=gr.themes.Monochrome(), title=f"Port from Python to {language}") as ui:
    with gr.Row(equal_height=True):
        with gr.Column(scale=6):
            python = gr.Code(
                label="Python (original)",
                value=python_hard,
                language="python",
                lines=26
            )
        with gr.Column(scale=6):
            cpp = gr.Code(
                label=f"{language} (generated)",
                value="",
                language="cpp",
                lines=26
            )

    with gr.Row(elem_classes=["controls"]):
        python_run = gr.Button("Run Python", elem_classes=["run-btn", "py"])
        model = gr.Dropdown(choices=models, value=models[0], show_label=False, interactive=True)
        convert = gr.Button(f"Port to {language}", elem_classes=["convert-btn"])
        cpp_run = gr.Button(f"Run {language}", elem_classes=["run-btn", "cpp"])

    with gr.Row(equal_height=True):
        with gr.Column(scale=6):
            python_out = gr.TextArea(label="Python result", lines=8, elem_classes=["py-out"])
        with gr.Column(scale=6):
            cpp_out = gr.TextArea(label=f"{language} result", lines=8, elem_classes=["cpp-out"])

    convert.click(fn=port, inputs=[model, python], outputs=[cpp])
    python_run.click(fn=run_python, inputs=[python], outputs=[python_out])
    cpp_run.click(fn=compile_and_run, inputs=[cpp], outputs=[cpp_out])

ui.launch(inbrowser=True)


/tmp/ipykernel_8175/413207069.py:3: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(css=CSS, theme=gr.themes.Monochrome(), title=f"Port from Python to {language}") as ui:


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


## RESULTS!

Qwen 2.5 Coder: FAIL  
Gemini 2.5 Pro: FAIL  
DeepSeek Coder v2: FAIL  
Qwen3 Coder 30B: FAIL  
Claude Sonnet 4.5: FAIL    
GPT-5: FAIL    

3rd place: GPT-oss-20B: 0.000341  
2nd place: Grok 4: 0.000317  
**1st place: OpenAI GPT-OSS 120B: 0.000304** 
google/gemma-4-31ba4 fail 
qwen/qwen3.5-35b-a3b fail
openai/gpt-oss-120b Execution Time: 0.000552 seconds

In [16]:
print(f"In Ed's experimenet, the GPT-OSS 120B model outcome is {33.755209/0.000304:,.0f} times faster than the Python code.")

In Ed's experimenet, the GPT-OSS 120B model outcome is 111,037 times faster than the Python code.


update.go:85: cannot change mount namespace according to change mount (/var/lib/snapd/hostfs/usr/local/share/doc /usr/local/share/doc none bind,ro 0 0): cannot write to "/var/lib/snapd/hostfs/usr/local/share/doc" because it would affect the host in "/var/lib/snapd"
update.go:85: cannot change mount namespace according to change mount (/var/lib/snapd/hostfs/usr/share/gimp/2.0/help /usr/share/gimp/2.0/help none bind,ro 0 0): cannot write to "/var/lib/snapd/hostfs/usr/share/gimp/2.0/help" because it would affect the host in "/var/lib/snapd"
update.go:85: cannot change mount namespace according to change mount (/var/lib/snapd/hostfs/usr/share/javascript /usr/share/javascript none bind,ro 0 0): cannot write to "/var/lib/snapd/hostfs/usr/share/javascript" because it would affect the host in "/var/lib/snapd"
update.go:85: cannot change mount namespace according to change mount (/var/lib/snapd/hostfs/usr/share/sphinx_rtd_theme /usr/share/sphinx_rtd_theme none bind,ro 0 0): cannot write to "/va

google/gemma-4-31b:2: Execution Time: 0.000411 seconds

Qwen/Qwen3.5-35B-A3B: Execution Time: 0.523256 seconds

openai/gpt-oss-120b:  Execution Time: 0.000595169 seconds